<a href="https://colab.research.google.com/github/kxilll/BSC_DPDM2025/blob/main/midterm663020286_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [271]:
import pandas as pd
import numpy as np

In [272]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [273]:
import os

BASE_PATH = "/content/drive/MyDrive/midterm bsc dpdm2025"

folders = os.listdir(BASE_PATH)
folders

['Output - (Clean monthly rain - 2025)',
 'Daily rain - (Statistics 2012-2024)',
 'Input - (Operation Daily rain from API_ 2025)']

In [274]:
INPUT_PATH = [f for f in folders if f.startswith("Input -")][0]
HIST_PATH  = [f for f in folders if f.startswith("Daily rain -")][0]
OUTPUT_PATH = [f for f in folders if f.startswith("Output -")][0]

INPUT_PATH = f"{BASE_PATH}/{INPUT_PATH}"
HIST_PATH = f"{BASE_PATH}/{HIST_PATH}"
OUTPUT_PATH = f"{BASE_PATH}/{OUTPUT_PATH}"

INPUT_PATH, HIST_PATH, OUTPUT_PATH

('/content/drive/MyDrive/midterm bsc dpdm2025/Input - (Operation Daily rain from API_ 2025)',
 '/content/drive/MyDrive/midterm bsc dpdm2025/Daily rain - (Statistics 2012-2024)',
 '/content/drive/MyDrive/midterm bsc dpdm2025/Output - (Clean monthly rain - 2025)')

In [275]:
os.listdir(INPUT_PATH)

['Stations in the model_HII.csv',
 '2025-10',
 '2025-07',
 '2025-11',
 '2025-12',
 '2025-09',
 '2025-08']

In [276]:
import pandas as pd

stations = pd.read_csv(f"{INPUT_PATH}/Stations in the model_HII.csv")
stations.head()

,Node,Lat,Long
0,ACRU,15.788054,104.642235
1,BBHN,9.536913,98.579490
2,BBUA,17.372976,103.984436
3,BCNG,17.386840,103.290870
4,BDAR,14.587073,102.495190


In [277]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 385 entries, 0 to 384
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Node    385 non-null    object 
 1   Lat     385 non-null    float64
 2   Long    385 non-null    float64
dtypes: float64(2), object(1)
memory usage: 9.2+ KB


In [278]:
from glob import glob

daily_files = glob(f"{INPUT_PATH}/2025-*/**/*.csv", recursive=True)
len(daily_files)

187

In [280]:
daily = pd.concat(
    [pd.read_csv(f) for f in daily_files],
    ignore_index=True
)

daily['rainfall_datetime'] = pd.to_datetime(daily['rainfall_datetime'])
daily.head()

,rainfall_datetime,rainfall_datetime_calc,rainfall_value,tele_station_id,tele_station_oldcode,measure_datetime,station_code,value
0,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,0.0,384.0,WEI062,2025-10-11,NaN,NaN
1,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,3.0,1109589.0,ONE100,2025-10-11,NaN,NaN
2,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,0.4,326.0,PHON,2025-10-11,NaN,NaN
3,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,0.0,65.0,SBT1,2025-10-11,NaN,NaN
4,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,5.8,1132568.0,ONE109,2025-10-11,NaN,NaN


In [282]:
daily["year"] = daily["rainfall_datetime"].dt.year
daily["month"] = daily["rainfall_datetime"].dt.month

daily.head()

,rainfall_datetime,rainfall_datetime_calc,rainfall_value,tele_station_id,tele_station_oldcode,measure_datetime,station_code,value,year,month
0,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,0.0,384.0,WEI062,2025-10-11,NaN,NaN,2025.0,10.0
1,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,3.0,1109589.0,ONE100,2025-10-11,NaN,NaN,2025.0,10.0
2,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,0.4,326.0,PHON,2025-10-11,NaN,NaN,2025.0,10.0
3,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,0.0,65.0,SBT1,2025-10-11,NaN,NaN,2025.0,10.0
4,2025-10-12 00:00:00+07:00,2025-10-11T07:00:00+07:00,5.8,1132568.0,ONE109,2025-10-11,NaN,NaN,2025.0,10.0


In [283]:
monthly = (
    daily
    .groupby(
        ["tele_station_oldcode", "year", "month"],
        as_index=False
    )
    .agg(
        monthly_rainfall_mm=("rainfall_value", "sum"),
        n_days=("rainfall_value", "count")
    )
)

monthly.head()

,tele_station_oldcode,year,month,monthly_rainfall_mm,n_days
0,ABRT,2025.0,10.0,7.2,7
1,ABRT,2025.0,11.0,31.8,30
2,ABRT,2025.0,12.0,0.0,31
3,ABRT,2026.0,1.0,0.0,1
4,ACRU,2025.0,7.0,83.0,30


In [284]:
monthly.describe()

,year,month,monthly_rainfall_mm,n_days
count,7125.000000,7125.000000,7125.000000,7125.000000
mean,2025.137123,8.358737,85.784421,23.581333
std,0.344001,3.313217,115.493891,11.918637
min,2025.000000,1.000000,0.000000,1.000000
25%,2025.000000,7.000000,1.000000,23.000000
50%,2025.000000,9.000000,48.600000,30.000000
75%,2025.000000,11.000000,131.800000,31.000000
max,2026.000000,12.000000,1535.400000,31.000000


In [285]:
monthly_geo = monthly.merge(
    stations,
    left_on="tele_station_oldcode",
    right_on="Node",
    how="inner"   # สำคัญมาก
)

monthly_geo.head()

,tele_station_oldcode,year,month,monthly_rainfall_mm,n_days,Node,Lat,Long
0,ACRU,2025.0,7.0,83.0,30,ACRU,15.788054,104.642235
1,ACRU,2025.0,8.0,164.6,31,ACRU,15.788054,104.642235
2,ACRU,2025.0,9.0,223.8,30,ACRU,15.788054,104.642235
3,ACRU,2025.0,10.0,72.0,31,ACRU,15.788054,104.642235
4,ACRU,2025.0,11.0,53.6,30,ACRU,15.788054,104.642235


In [286]:
monthly_geo.isna().sum()

,0
tele_station_oldcode,0
year,0
month,0
monthly_rainfall_mm,0
n_days,0
Node,0
Lat,0
Long,0


In [287]:
final = monthly_geo[[
    "tele_station_oldcode",
    "Lat",
    "Long",
    "year",
    "month",
    "monthly_rainfall_mm"
]].copy()

final.rename(columns={
    "tele_station_oldcode": "station_code",
    "monthly_rainfall_mm": "rainfall_mm"
}, inplace=True)

final.head()

,station_code,Lat,Long,year,month,rainfall_mm
0,ACRU,15.788054,104.642235,2025.0,7.0,83.0
1,ACRU,15.788054,104.642235,2025.0,8.0,164.6
2,ACRU,15.788054,104.642235,2025.0,9.0,223.8
3,ACRU,15.788054,104.642235,2025.0,10.0,72.0
4,ACRU,15.788054,104.642235,2025.0,11.0,53.6


In [288]:
final = final.sort_values(
    by=["station_code", "year", "month"]
).reset_index(drop=True)

final.head()

,station_code,Lat,Long,year,month,rainfall_mm
0,ACRU,15.788054,104.642235,2025.0,7.0,83.0
1,ACRU,15.788054,104.642235,2025.0,8.0,164.6
2,ACRU,15.788054,104.642235,2025.0,9.0,223.8
3,ACRU,15.788054,104.642235,2025.0,10.0,72.0
4,ACRU,15.788054,104.642235,2025.0,11.0,53.6


In [289]:
output_file = f"{OUTPUT_PATH}/output_monthly_rainfall.csv"
final.to_csv(output_file, index=False)

output_file

'/content/drive/MyDrive/midterm bsc dpdm2025/Output - (Clean monthly rain - 2025)/output_monthly_rainfall.csv'

In [290]:
# ==============================
# STEP 1: Create full daily calendar (2025)
# ==============================

full_dates = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="D"
)

stations_list = stations["Node"].unique()

calendar = pd.MultiIndex.from_product(
    [stations_list, full_dates],
    names=["tele_station_oldcode", "rainfall_datetime"]
).to_frame(index=False)

calendar.head()

,tele_station_oldcode,rainfall_datetime
0,ACRU,2025-01-01
1,ACRU,2025-01-02
2,ACRU,2025-01-03
3,ACRU,2025-01-04
4,ACRU,2025-01-05


In [291]:
# ==============================
# STEP 2: Detect missing records
# ==============================

daily['rainfall_datetime'] = daily['rainfall_datetime'].dt.tz_localize(None)

daily_full = calendar.merge(
    daily,
    on=["tele_station_oldcode", "rainfall_datetime"],
    how="left"
)

missing_summary = (
    daily_full
    .groupby("tele_station_oldcode")
    .agg(
        total_days=("rainfall_datetime", "count"),
        observed_days=("rainfall_value", "count")
    )
)

missing_summary["missing_days"] = (
    missing_summary["total_days"] - missing_summary["observed_days"]
)

missing_summary["missing_ratio"] = (
    missing_summary["missing_days"] / missing_summary["total_days"]
)

missing_summary.sort_values("missing_ratio", ascending=False).head()

,total_days,observed_days,missing_days,missing_ratio
tele_station_oldcode,,,,
BKK009,365,0,365,1.000000
NPI002,365,15,350,0.958904
WAN005,365,36,329,0.901370
CHI010,365,42,323,0.884932
CHNA,365,45,320,0.876712


In [292]:
# ==============================
# STEP 3: Filter stations (≤ 20% missing)
# ==============================

valid_stations = missing_summary[
    missing_summary["missing_ratio"] <= 0.20
].index

daily_full = daily_full[
    daily_full["tele_station_oldcode"].isin(valid_stations)
]

In [294]:
print("จำนวนสถานีทั้งหมด:", missing_summary.shape[0])
print("จำนวนสถานีที่ผ่านเกณฑ์ ≤20%:", len(valid_stations))

missing_summary.sort_values("missing_ratio").head(10)

จำนวนสถานีทั้งหมด: 385
จำนวนสถานีที่ผ่านเกณฑ์ ≤20%: 0


,total_days,observed_days,missing_days,missing_ratio
tele_station_oldcode,,,,
BJIG,365,183,182,0.49863
BHUN,365,183,182,0.49863
BGRT,365,183,182,0.49863
BDMG,365,183,182,0.49863
BDLH,365,183,182,0.49863
BDGN,365,183,182,0.49863
BDAR,365,183,182,0.49863
BCNG,365,183,182,0.49863
BBHN,365,183,182,0.49863


In [249]:
import os

BASE_PATH = "/content/drive/MyDrive/midterm bsc dpdm2025"
HIST_DIR = f"{BASE_PATH}/Daily rain - (Statistics 2012-2024)"

os.listdir(HIST_DIR)

['hii_daily_rain.csv']

In [250]:
historical.head()

,station_code,measure_datetime,data,station_name,latitude,longitude,quality_flag,basin,sub_basin,tambon,amphoe,province,data_type,doy
0,ABRT,2012-01-01 07:00:00+07:00,0.0,วัดเวฬุวัน,16.054785,103.66429,U,ชี,ลำน้ำชีส่วนที่ 4/2,เหนือเมือง,เมืองร้อยเอ็ด,ร้อยเอ็ด,rainfall_daily,1
1,ABRT,2012-01-03 07:00:00+07:00,0.0,วัดเวฬุวัน,16.054785,103.66429,U,ชี,ลำน้ำชีส่วนที่ 4/2,เหนือเมือง,เมืองร้อยเอ็ด,ร้อยเอ็ด,rainfall_daily,3
2,ABRT,2012-01-06 07:00:00+07:00,0.0,วัดเวฬุวัน,16.054785,103.66429,U,ชี,ลำน้ำชีส่วนที่ 4/2,เหนือเมือง,เมืองร้อยเอ็ด,ร้อยเอ็ด,rainfall_daily,6
3,ABRT,2012-01-07 07:00:00+07:00,0.0,วัดเวฬุวัน,16.054785,103.66429,U,ชี,ลำน้ำชีส่วนที่ 4/2,เหนือเมือง,เมืองร้อยเอ็ด,ร้อยเอ็ด,rainfall_daily,7
4,ABRT,2012-01-08 07:00:00+07:00,0.0,วัดเวฬุวัน,16.054785,103.66429,U,ชี,ลำน้ำชีส่วนที่ 4/2,เหนือเมือง,เมืองร้อยเอ็ด,ร้อยเอ็ด,rainfall_daily,8


In [251]:
historical = pd.read_csv(f"{HIST_DIR}/hii_daily_rain.csv")
historical.columns

Index(['station_code', 'measure_datetime', 'data', 'station_name', 'latitude',
       'longitude', 'quality_flag', 'basin', 'sub_basin', 'tambon', 'amphoe',
       'province', 'data_type'],
      dtype='object')

In [189]:
historical.columns

Index(['station_code', 'measure_datetime', 'data', 'station_name', 'latitude',
       'longitude', 'quality_flag', 'basin', 'sub_basin', 'tambon', 'amphoe',
       'province', 'data_type'],
      dtype='object')

In [212]:
# ==============================
# STEP 4: Median imputation by Day-of-Year (from historical)
# ==============================

# 4.1 อ่านข้อมูล historical
historical = pd.read_csv(f"{HIST_DIR}/hii_daily_rain.csv")

# แปลง datetime
historical["measure_datetime"] = pd.to_datetime(historical["measure_datetime"])

# day of year
historical["doy"] = historical["measure_datetime"].dt.dayofyear

# 4.2 คำนวณ median แยกตามสถานี + day of year
median_table = (
    historical
    .groupby(["station_code", "doy"])["data"]
    .median()
    .reset_index()
    .rename(columns={"data": "median_rain"})
)

median_table.head()

,station_code,doy,median_rain
0,ABRT,1,0.0
1,ABRT,2,0.0
2,ABRT,3,0.0
3,ABRT,4,0.0
4,ABRT,5,0.0


In [252]:
daily_full.columns

Index(['tele_station_oldcode', 'rainfall_datetime', 'rainfall_datetime_calc',
       'rainfall_value', 'tele_station_id', 'measure_datetime', 'station_code',
       'value', 'year', 'month'],
      dtype='object')

In [253]:
daily_full = daily_full.drop(
    columns=[c for c in daily_full.columns if c in ["station_code", "station_code_x", "station_code_y", "median_rain"]],
    errors="ignore"
)

In [254]:
daily_full["doy"] = daily_full["rainfall_datetime"].dt.dayofyear
daily_full = daily_full.merge(
    median_table,
    left_on=["tele_station_oldcode", "doy"],
    right_on=["station_code", "doy"],
    how="left"
)

In [255]:
daily_full["rainfall_value"] = daily_full["rainfall_value"].fillna(
    daily_full["median_rain"]
)

In [256]:
daily_full = daily_full.drop(columns=["median_rain", "station_code"])

In [257]:
daily_full.isna().sum()

,0
tele_station_oldcode,0
rainfall_datetime,0
rainfall_datetime_calc,0
rainfall_value,0
tele_station_id,0
measure_datetime,0
value,0
year,0
month,0
doy,0


In [259]:
daily = daily_full.copy()

In [261]:
print("monthly rows:", len(monthly))
print("monthly_geo rows:", len(monthly_geo))
print("final rows:", len(final))

monthly rows: 7125
monthly_geo rows: 2580
final rows: 2580


In [262]:
len(valid_stations)

0

In [263]:
missing_summary.describe()

,total_days,observed_days,missing_days,missing_ratio
count,385.0,385.000000,385.000000,385.000000
mean,365.0,171.184416,193.815584,0.531002
std,0.0,31.374083,31.374083,0.085956
min,365.0,0.000000,182.000000,0.498630
25%,365.0,181.000000,182.000000,0.498630
50%,365.0,183.000000,182.000000,0.498630
75%,365.0,183.000000,184.000000,0.504110
max,365.0,183.000000,365.000000,1.000000


In [264]:
missing_summary.sort_values("missing_ratio", ascending=False).head(10)

,total_days,observed_days,missing_days,missing_ratio
tele_station_oldcode,,,,
BKK009,365,0,365,1.000000
NPI002,365,15,350,0.958904
WAN005,365,36,329,0.901370
CHI010,365,42,323,0.884932
CHNA,365,45,320,0.876712
BPTG,365,50,315,0.863014
BKK017,365,52,313,0.857534
PSLT,365,54,311,0.852055
YCUM,365,55,310,0.849315


In [265]:
# ============================
# STEP 5: Aggregate to monthly rainfall
# ============================

daily["year"] = daily["rainfall_datetime"].dt.year
daily["month"] = daily["rainfall_datetime"].dt.month

monthly = (
    daily
    .groupby(
        ["tele_station_oldcode", "year", "month"],
        as_index=False
    )
    .agg(
        monthly_rainfall_mm=("rainfall_value", "sum"),
        n_days=("rainfall_value", "count")
    )
)

monthly.head()

,tele_station_oldcode,year,month,monthly_rainfall_mm,n_days


In [266]:
# ============================
# STEP 6: Merge station coordinates
# ============================

monthly_geo = monthly.merge(
    stations,
    left_on="tele_station_oldcode",
    right_on="Node",
    how="inner"
)

monthly_geo.head()

,tele_station_oldcode,year,month,monthly_rainfall_mm,n_days,Node,Lat,Long


In [307]:
# ============================
# STEP 7: Final clean output (NO Lat / Long)
# ============================

final = monthly[[
    "tele_station_oldcode",
    "monthly_rainfall_mm",
    "month",
    "year"
]].copy()

final = final.rename(columns={
    "tele_station_oldcode": "station_code",
    "monthly_rainfall_mm": "rainfall_mm"
})

final = final.sort_values(
    by=["station_code", "year", "month"]
).reset_index(drop=True)

final.head()

,station_code,rainfall_mm,month,year
0,ABRT,7.2,10.0,2025.0
1,ABRT,31.8,11.0,2025.0
2,ABRT,0.0,12.0,2025.0
3,ABRT,0.0,1.0,2026.0
4,ACRU,83.0,7.0,2025.0


In [308]:
output_file = f"{OUTPUT_PATH}/output_monthly_rainfall.csv"
final.to_csv(output_file, index=False)
output_file

'/content/drive/MyDrive/midterm bsc dpdm2025/Output - (Clean monthly rain - 2025)/output_monthly_rainfall.csv'

In [296]:
# Pipeline summary:
# 1) Create full daily calendar (station × date)
# 2) Detect missing daily rainfall
# 3) Filter stations with missing ratio ≤ 20%
# 4) Impute missing rainfall using median by day-of-year
# 5) Aggregate to monthly rainfall

In [309]:
excluded_stations = missing_summary[
    missing_summary["missing_ratio"] > 0.20
].index

print("Excluded stations:", list(excluded_stations))
print("Remaining stations:", len(valid_stations))

Excluded stations: ['ACRU', 'BBHN', 'BBUA', 'BCNG', 'BDAR', 'BDCP', 'BDGN', 'BDLH', 'BDMG', 'BGRT', 'BHMN', 'BHRA', 'BHUN', 'BJIG', 'BKDN', 'BKHL', 'BKHN', 'BKK001', 'BKK008', 'BKK009', 'BKK015', 'BKK017', 'BKK018', 'BKK019', 'BKUG', 'BKWN', 'BLAT', 'BLKO', 'BLUG', 'BMDG', 'BMNK', 'BNAN', 'BNGR', 'BNHG', 'BNHI', 'BNHO', 'BNKN', 'BNKP', 'BNLG', 'BNMK', 'BNPI', 'BNPU', 'BNTK', 'BOKA', 'BPIA', 'BPK004', 'BPLA', 'BPPS', 'BPTG', 'BRKM', 'BSBA', 'BSDE', 'BSKJ', 'BSMP', 'BSPN', 'BSUM', 'BTHO', 'BWKG', 'BYNU', 'CATK', 'CBUR', 'CCSS', 'CEHM', 'CGDO', 'CGKM', 'CGMN', 'CGSN', 'CHHA', 'CHI005', 'CHI010', 'CHI011', 'CHI012', 'CHI013', 'CHM001', 'CHM002', 'CHM003', 'CHM004', 'CHN004', 'CHNA', 'CHPA', 'CHR003', 'CHR004', 'CHR005', 'CHUN', 'CLPK', 'CMSG', 'CPKC', 'CPY003', 'CPY004', 'CPY009', 'CPY010', 'CPY012', 'CPY013', 'CPY015', 'CPY017', 'CSMO', 'CTKN', 'DCCT', 'DGSL', 'DITO', 'DIV001', 'DIV002', 'DIV003', 'DIV004', 'DIV005', 'DKLG', 'DKTI', 'DMKH', 'DNCM', 'DSKT', 'DTAN', 'HCTT', 'HKTG', 'HNKA', 

In [313]:
final = final[[
    "station_code",
    "rainfall_mm",
    "year",
    "month"
]]

In [314]:
final["station_code"] = final["station_code"].astype(str)
final["rainfall_mm"] = final["rainfall_mm"].astype(float)
final["year"] = final["year"].astype(int)
final["month"] = final["month"].astype(int)

In [315]:
final = final.sort_values(
    by=["station_code", "year", "month"]
).reset_index(drop=True)

In [316]:
final.head(20)
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7125 entries, 0 to 7124
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   station_code  7125 non-null   object 
 1   rainfall_mm   7125 non-null   float64
 2   year          7125 non-null   int64  
 3   month         7125 non-null   int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 222.8+ KB


In [302]:
output_file = f"{OUTPUT_PATH}/output_monthly_rainfall.csv"
final.to_csv(output_file, index=False)

output_file

'/content/drive/MyDrive/midterm bsc dpdm2025/Output - (Clean monthly rain - 2025)/output_monthly_rainfall.csv'

In [303]:
stations.columns

Index(['Node', 'Lat', 'Long'], dtype='object')

In [306]:
final = final[
    ["station_code", "year", "month", "rainfall_mm"]
]


### อภิปราย : เหตุใดจึงเลือกใช้ Median แทน Mean

ข้อมูลปริมาณน้ำฝนมีลักษณะการกระจายที่ไม่สมมาตร (skewed distribution)
และมักพบค่าที่สูงมากผิดปกติจากเหตุการณ์ฝนตกหนักในบางวัน
หากใช้ค่าเฉลี่ย (Mean) ค่าดังกล่าวจะได้รับอิทธิพลจากค่าผิดปกติ (outliers)
ทำให้ค่าที่นำไปใช้เติมข้อมูลไม่สะท้อนลักษณะของข้อมูลโดยทั่วไป

ค่า Median มีความทนทานต่อค่าผิดปกติ และสามารถแทนค่ากลางของข้อมูลได้ดีกว่า
โดยเฉพาะเมื่อนำมาใช้เติมข้อมูลที่หายไปตาม Day of Year
จึงเหมาะสมกับข้อมูลปริมาณน้ำฝนมากกว่าการใช้ค่า Mean


###ไฟล์ผลลัพธ์ข้อมูลรายเดือน
https://drive.google.com/file/d/1bvoiAPAPbjLV1pDwmA2F38hWTVNrmW5c/view?usp=sharing

###บันทึกการใช้ AI
https://docs.google.com/document/d/15fB1inYkEZOrqnTTCJN7Fi-0gXDSlo94yoD2Y6OP5f4/edit?usp=sharing